### Data Download & Preparation: Cleaning & Filtering IT Resumes

We will first,   
- Download the Kaggle Resume dataset. 

Then,
-  filters for IT & Tech roles, parses raw HTML using BeautifulSoup, and exports a clean dataset for the matching engine.

In [2]:
# Import necessary libraries
import os
import re
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import kagglehub

# setting for inspecting long text strings
pd.set_option("display.max_colwidth", 200)

In [ ]:
# folder for raw data
raw_dir = Path("raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# download the dataset from Kaggle
print("Downloading snehaanbhawal/resume-dataset via kagglehub...")
print("=="*40)

dataset_dir = kagglehub.dataset_download("raw")
print(f"Dataset downloaded to local cache: {dataset_dir}")

In [7]:
dataset_dir = "raw"
dataset_path = Path(dataset_dir)
csv_files = list(dataset_path.rglob("*.csv"))

print("CSV files found in downloaded data directory:")
for f in csv_files:
    print(f" - {f.name}")


csv_file_path = csv_files[0]

CSV files found in downloaded data directory:
 - Resume.csv


In [8]:
df = pd.read_csv(csv_file_path)

print(f"Total rows loaded: {len(df)}")
df.head(3)

Total rows loaded: 2484


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. ...,"<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME500375979"" style=""\n padding-top:0px;\n ""> <div class...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS Summary Versatile media professional with background in Communications, Marketing, Human Resources and Technology. Experience 09/201...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME911808366"" style=""padding-top:0px;""> <div class=""paragraph PA...",HR
2,33176873,"HR DIRECTOR Summary Over 20 years experience in recruiting, 15 plus years in Human Resources Executive Management, 5 years of HRIS development and maintenance 4 years work...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME1008511259"" style=""padding-top:0px;""> <div class=""paragraph P...",HR


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           2484 non-null   int64
 1   Resume_str   2484 non-null   str  
 2   Resume_html  2484 non-null   str  
 3   Category     2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 77.8 KB


In [10]:
print("Top 15 resume categories:")
print(df["Category"].value_counts().head(15))

Top 15 resume categories:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
Name: count, dtype: int64


### Filtering IT & Tech Categories
Separating the resumes belonging to IT, Software Engineering, and Digital Media to build an IT-specific benchmark dataset.

In [11]:
it_categories = ["INFORMATION-TECHNOLOGY", "ENGINEERING", "DIGITAL-MEDIA"]

# Filter dataset
it_df = df[df["Category"].isin(it_categories)].copy().reset_index(drop=True)

print(f"Filtered down from {len(df)} to {len(it_df)} IT resumes.")
print("\nBreakdown by selected category:")
print(it_df["Category"].value_counts())

Filtered down from 2484 to 334 IT resumes.

Breakdown by selected category:
Category
INFORMATION-TECHNOLOGY    120
ENGINEERING               118
DIGITAL-MEDIA              96
Name: count, dtype: int64


### HTML Cleaning & Text Normalization
The `Resume_str` column contains raw string formatting issues (like merged headings and &`nbsp;` remnants). Parsing `Resume_html` with BeautifulSoup provides clean word boundaries.

In [12]:
def clean_resume_html(html_text):
    if pd.isna(html_text) or not str(html_text).strip():
        return ""

    # Strip HTML tags 
    soup = BeautifulSoup(str(html_text), "html.parser")
    text = soup.get_text(separator=" ")

    # Normalize whitespace & remove HTML tags
    text = text.replace("\xa0", " ").replace("&nbsp;", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()

In [13]:
print("Cleaning HTML content and normalizing text...")
it_df["clean_text"] = it_df["Resume_html"].apply(clean_resume_html)

# fallback for empty string
empty_mask = it_df["clean_text"].str.len() == 0
if empty_mask.sum() > 0:
    print(f"Applying string fallback for {empty_mask.sum()} empty entries...")
    it_df.loc[empty_mask, "clean_text"] = it_df.loc[empty_mask, "Resume_str"]

Cleaning HTML content and normalizing text...


In [14]:
it_df["word_count"] = it_df["clean_text"].apply(lambda x: len(x.split()))

final_df = (
    it_df[["ID", "Category", "clean_text", "word_count"]]
    .drop_duplicates(subset=["ID"])
    .reset_index(drop=True)
)

print(f"Final clean dataset shape: {final_df.shape}")
final_df.head(3)

Final clean dataset shape: (334, 4)


,ID,Category,clean_text,word_count
0,36856210,INFORMATION-TECHNOLOGY,INFORMATION TECHNOLOGY Summary Dedicated Information Assurance Professional well-versed in analyzing and mitigating risk and finding cost-effective solutions. Excels at boosting performance and pr...,617
1,21780877,INFORMATION-TECHNOLOGY,"INFORMATION TECHNOLOGY SPECIALIST GS11 Experience 07/2004 to Current Information Technology Specialist GS11 Company Name － City , State Information Technology Specialist; Supervison; Project Manag...",807
2,33241454,INFORMATION-TECHNOLOGY,"INFORMATION TECHNOLOGY SUPERVISOR Summary Seeking a position as an Information Technology Specialist. Over 5 years of information technology experience in the U.S. Army, including over 1 year of s...",540


### Now, Save final IT related resume dataset


In [15]:
output_dir = Path("processed")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "final_resumes.csv"
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved processed dataset to {output_file.resolve()}")

Saved processed dataset to /Users/sujansharma/Documents/0Study_Files/Python-Programming/ResuMatch/data/data_preparation/processed/final_resumes.csv
